In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║ Onset-boundary sensitivity analysis                                 ║
# ║ 각 위기의 시작연도만 -2,-1,0,+1,+2년 이동 / 종료연도는 고정        ║
# ║ 모델 1·2·3을 매번 다시 학습하여 LOO-CV AUC 계산                     ║
# ╚══════════════════════════════════════════════════════════════════════╝

import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import roc_auc_score

# ── 필요 변수 ─────────────────────────────────────────
# 앞 셀에서 이미 있어야 하는 것:
# df_raw, y_bin, X_full
#
# Model 3 재구성을 위해 원본 master CSV 필요:
RAW_MASTER_PATH = "kangxi_master_A_1661_1722.csv"   # ← 경로 맞게 수정

years = np.array(df_raw.index.astype(int))
y_base = pd.Series(y_bin.astype(int), index=years)

# ── Model 3용 raw master 로드 ────────────────────────
raw_master = pd.read_csv(RAW_MASTER_PATH, encoding="utf-8-sig", index_col=0, dtype=str)
raw_master.index = raw_master.index.astype(str)

PARTICLES = set([
    "之","乎","者","也","而","以","於","其","為","與","乃","則","所","矣",
    "焉","哉","夫","且","若","如","兮","耳","爾","耶","邪","歟",
    "不","無","非","莫","勿","毋","弗",
    "此","彼","是","茲","斯","我","汝",
    "皆","亦","即","各","又","更","已","故","因","遂","從","及",
    "或","仍","再","復","旋","頓","俱","並","并","尚","猶",
    "自","至","由","用","共","同","均","咸","悉",
    "等","諸","餘","凡","每","盡","全",
    "一","二","三","四","五","六","七","八","九","十","百","千","萬","億","兩","半",
    "上","下","前","後","中","內","外","左","右","東","西","南","北",
    "今","昔","先","初","末","始","終","早","晚","久","頃",
    "有","曰","云","言","稱","謂","令","使","命","允","欽","奉","遵",
    "奏","覆","議","准","照","依",
    "大","小","多","少","新","舊","正","副","主","次","高","低",
    "都","公","行","事","日","月","年","時","處","地","人","王",
    "臣","民","兵","官","子","父","母","兄","弟","姪","孫",
    "위기"
])

# ── 1. 현재 위기구간(block) 자동 추출 ─────────────────
def contiguous_blocks(y_series):
    years = y_series.index.to_list()
    vals  = y_series.values.tolist()
    blocks = []
    in_block = False
    prev = None

    for yr, v in zip(years, vals):
        if v == 1 and not in_block:
            start = yr
            in_block = True
        elif v == 0 and in_block:
            blocks.append((start, prev))
            in_block = False
        prev = yr

    if in_block:
        blocks.append((start, prev))

    return blocks

base_blocks = contiguous_blocks(y_base)
print("기준 위기구간:", base_blocks)

# ── 2. 시작연도만 이동시키는 함수 ─────────────────────
def shift_onsets_only(blocks, years, delta, clamp_to_end=True):
    """
    blocks: [(start, end), ...]
    delta:  시작연도 이동량
    종료연도는 고정
    clamp_to_end=True이면 start+delta > end 일 때 start=end 로 클리핑
    """
    min_year = int(np.min(years))
    max_year = int(np.max(years))
    out = pd.Series(0, index=years, dtype=int)
    shifted_blocks = []

    for start, end in blocks:
        new_start = start + delta
        new_start = max(min_year, new_start)

        if clamp_to_end:
            new_start = min(end, new_start)

        if new_start <= end:
            out.loc[new_start:end] = 1
            shifted_blocks.append((new_start, end))

    return out, shifted_blocks

# ── 3. LOO-CV AUC 헬퍼 ────────────────────────────────
def loo_auc(model, X, y):
    proba = np.zeros(len(y), dtype=float)
    loo = LeaveOneOut()

    for tr, te in loo.split(X):
        m = copy.deepcopy(model)
        m.fit(X[tr], y[tr])
        proba[te] = m.predict_proba(X[te])[:, 1]

    return roc_auc_score(y, proba)

# ── 4. Model 1: Top-50 correlated features ───────────
def model1_auc_with_shift(y_shift):
    scaler = StandardScaler()
    Xs = pd.DataFrame(
        scaler.fit_transform(X_full),
        columns=X_full.columns,
        index=X_full.index
    )

    corr_y = Xs.corrwith(pd.Series(y_shift.astype(float), index=Xs.index))
    top50 = corr_y.abs().nlargest(50).index
    X1_s = Xs[top50].values

    lr1_s = LogisticRegression(
        penalty="l2", C=0.05, max_iter=3000,
        class_weight="balanced", solver="lbfgs", random_state=42
    )
    return loo_auc(lr1_s, X1_s, y_shift)

# ── 5. Model 2: TF-IDF + PCA(20) ─────────────────────
def model2_auc_with_shift(y_shift):
    tfidf = TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True)
    X2_t = tfidf.fit_transform(X_full.values).toarray()

    pca = PCA(n_components=20, random_state=42)
    X2_s = pca.fit_transform(X2_t)

    lr2_s = LogisticRegression(
        penalty="l2", C=1.0, max_iter=3000,
        class_weight="balanced", random_state=42
    )
    return loo_auc(lr2_s, X2_s, y_shift)

# ── 6. Model 3: rare-word<=10 제거 + TF-IDF + PCA(20) ─
def model3_auc_with_shift(y_shift):
    # raw_master: (단어 × 연도) → transpose → (연도 × 단어)
    df_t3 = raw_master.replace("NULL", 0).fillna(0).astype(int).T.copy()
    df_t3.index = df_t3.index.astype(int)

    # shifted label 덮어쓰기
    df_t3["위기"] = pd.Series(y_shift, index=df_t3.index).astype(int)

    # 조사/허사 제거
    drop_cols = [c for c in df_t3.columns if c in PARTICLES]
    df_t3 = df_t3.drop(columns=drop_cols, errors="ignore")

    # rare words <=10 제거
    X3_raw = df_t3.drop(columns=["위기"], errors="ignore").fillna(0).astype(float)
    rare_cols = X3_raw.columns[X3_raw.sum(axis=0) <= 10].tolist()
    X3_raw = X3_raw.drop(columns=rare_cols, errors="ignore")

    y3 = pd.Series(y_shift, index=df_t3.index).astype(int).values

    tfidf3 = TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True)
    X3_t = tfidf3.fit_transform(X3_raw.values).toarray()

    pca3 = PCA(n_components=20, random_state=42)
    X3_s = pca3.fit_transform(X3_t)

    lr3_s = LogisticRegression(
        penalty="l2", C=1.0, max_iter=3000,
        class_weight="balanced", random_state=42
    )
    return loo_auc(lr3_s, X3_s, y3)

# ── 7. Δ = -2, -1, 0, +1, +2 반복 ───────────────────
rows = []

for delta in [-2, -1, 0, 1, 2]:
    y_shifted, shifted_blocks = shift_onsets_only(base_blocks, years, delta, clamp_to_end=True)

    auc1 = model1_auc_with_shift(y_shifted.values)
    auc2 = model2_auc_with_shift(y_shifted.values)
    auc3 = model3_auc_with_shift(y_shifted.values)

    rows.append({
        "delta_year": delta,
        "n_positive": int(y_shifted.sum()),
        "model1_auc": auc1,
        "model2_auc": auc2,
        "model3_auc": auc3,
        "blocks": "; ".join([f"{s}-{e}" if s != e else str(s) for s, e in shifted_blocks]),
    })

    print(f"Δ={delta:+d} | n_pos={int(y_shifted.sum())} | "
          f"M1={auc1:.3f} | M2={auc2:.3f} | M3={auc3:.3f}")

sens_df = pd.DataFrame(rows)
display(sens_df)

# 저장
sens_df.to_csv("kangxi_onset_sensitivity_auc.csv", index=False, encoding="utf-8-sig")

# ── 8. Plot ───────────────────────────────────────────
BG    = "#FAF7F0"
PANEL = "#F0EBE0"
GRID  = "#C8BFB0"
TM    = "#0F172A"
TS    = "#475569"

COL1 = "#1D4ED8"
COL2 = "#047857"
COL3 = "#B45309"

plt.figure(figsize=(10, 6), facecolor=BG)
ax = plt.gca()
ax.set_facecolor(PANEL)

for sp in ax.spines.values():
    sp.set_color(GRID)
    sp.set_linewidth(0.9)

ax.plot(sens_df["delta_year"], sens_df["model1_auc"],
        marker="o", markersize=7, lw=2.4, color=COL1, label="Model 1")
ax.plot(sens_df["delta_year"], sens_df["model2_auc"],
        marker="o", markersize=7, lw=2.4, color=COL2, label="Model 2")
ax.plot(sens_df["delta_year"], sens_df["model3_auc"],
        marker="o", markersize=7, lw=2.4, color=COL3, label="Model 3")

ax.axvline(0, color=TS, ls="--", lw=1, alpha=0.8)

for x, y in zip(sens_df["delta_year"], sens_df["model1_auc"]):
    ax.text(x, y + 0.015, f"{y:.3f}", color=COL1, fontsize=9, ha="center")
for x, y in zip(sens_df["delta_year"], sens_df["model2_auc"]):
    ax.text(x, y - 0.040, f"{y:.3f}", color=COL2, fontsize=9, ha="center")
for x, y in zip(sens_df["delta_year"], sens_df["model3_auc"]):
    ax.text(x, y + 0.015, f"{y:.3f}", color=COL3, fontsize=9, ha="center")

ax.set_xlabel("시작 경계 이동 Δ (년) — 종료연도는 고정", color=TM, fontsize=11)
ax.set_ylabel("LOO-CV AUC", color=TM, fontsize=11)
ax.set_title(
    "Onset-boundary sensitivity analysis\n"
    "(각 위기의 시작연도만 -2~-+2년 이동, 종료연도 고정)",
    color=TM, fontsize=13, fontweight="bold"
)
ax.set_xticks([-2, -1, 0, 1, 2])
ax.set_ylim(0.25, 1.05)
ax.grid(axis="y", color=GRID, ls="--", lw=0.6, alpha=0.8)
ax.tick_params(colors=TM)

leg = ax.legend(frameon=True, facecolor=BG, edgecolor=GRID, fontsize=10)
for t in leg.get_texts():
    t.set_color(TM)

note = "Note: start+Δ > end 인 경우, onset은 end year로 clip(start=min(end, start+Δ)) 처리"
ax.text(0.01, -0.18, note, transform=ax.transAxes, fontsize=9, color=TS)

plt.tight_layout()
plt.savefig("kangxi_onset_sensitivity_auc.png", dpi=180, bbox_inches="tight", facecolor=BG)
plt.show()
